# Stelow cohort — patient MHC II classification

**Goal:** Derive and validate MHC II patient classifications for the Stelow
cohort using multiplexed immunofluorescence data from two independent panels
(Panel 3: lymphocyte panel, Panel 4: APC panel). Classifications are based
on the fraction of PanCK+ tumor cells expressing MHC II per patient,
aggregated across all ROIs. A threshold of 0.4 is applied to define
MHC II high vs low groups. Discordant cases across panels are identified
and excluded from downstream analyses.

**Input:** Panel 3 and Panel 4 HALO summary CSVs, Stelow cohort metadata.

**Output:** Figure 5C — per-patient MHC II+ fraction across both panels
with classification threshold and S21 discordance highlighted.

In [ ]:
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

fig_out   = fig_dir   / 'figure5'
table_out = table_dir / 'figure5'

fig_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

In [ ]:
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

MHCII_THRESHOLD = 0.5


## Load data

Panel 3 and Panel 4 HALO summary CSVs are loaded and filtered to LUAD
adenocarcinoma patients. Per-patient MHC II+ fraction is computed from
aggregated tumor PanCK+MHCII+ and PanCK+MHCII- cell counts across all ROIs.

In [ ]:
panel3_path   = Path('/project/Dolatshahi_Lab/Gabe_Hanson/data/mhc2_vectra_data/object_data/stelow_slides/full_image_set/panel3/summary_data.csv')
panel4_path  = Path('/project/Dolatshahi_Lab/Gabe_Hanson/data/mhc2_vectra_data/object_data/stelow_slides/full_image_set/panel4/summary_data_pan4.csv')
metadata_path = Path('/project/Dolatshahi_Lab/Gabe_Hanson/stelow_moore/vectra_stelow_slide_metadata.csv')

metadata = pd.read_csv(metadata_path)
metadata_cols = ['Code', 'HIST']

def load_and_filter(path, metadata, metadata_cols):
    df = pd.read_csv(path)
    df['ROI']       = df['Image Location'].str.extract(r'(S\d{1,2} LUNG.*?\[.*?\])')
    df['PatientID'] = df['Image Location'].str.extract(r'(S\d{1,2} LUNG)')
    df['Code']      = df['PatientID'].str.replace(' LUNG', '', regex=False)
    df = df.merge(metadata[metadata_cols], on='Code', how='left')
    df = df.loc[df['HIST'] == 'Adenocarcinoma'].copy()
    return df

panel3 = load_and_filter(panel3_path, metadata, metadata_cols)
panel4 = load_and_filter(panel4_path, metadata, metadata_cols)

# derive MHCII+ and MHCII- totals for panel3
panel3['tumor: PanCK+MHCII+ Cells'] = (
    panel3['tumor: PanCK+MHCI-MHCII+ Cells'] +
    panel3['tumor: PanCK+MHCI+MHCII+ Cells']
)
panel3['tumor: PanCK+MHCII- Cells'] = (
    panel3['tumor: PanCK+MHCI+MHCII- Cells'] +
    panel3['tumor: PanCK+MHCI-MHCII- Cells']
)

def compute_mhc2_fraction(df):
    agg = (
        df.groupby('PatientID')[['tumor: PanCK+MHCII+ Cells', 'tumor: PanCK+MHCII- Cells']]
        .sum()
        .reset_index()
    )
    agg['mhc2_pos_fraction'] = (
        agg['tumor: PanCK+MHCII+ Cells'] /
        (agg['tumor: PanCK+MHCII+ Cells'] + agg['tumor: PanCK+MHCII- Cells'])
    )
    return agg[['PatientID', 'mhc2_pos_fraction']]

p3_fractions = compute_mhc2_fraction(panel3).rename(columns={'mhc2_pos_fraction': 'panel3_fraction'})
p4_fractions = compute_mhc2_fraction(panel4).rename(columns={'mhc2_pos_fraction': 'panel4_fraction'})

combined['classification'] = np.where(
    combined['panel4_fraction'] >= MHCII_THRESHOLD,
    'class II high', 'class II low'
)

# flag S21 as discordant for visualization purposes only — not excluded
combined['discordant'] = (
    (combined['panel3_fraction'] >= MHCII_THRESHOLD) != 
    (combined['panel4_fraction'] >= MHCII_THRESHOLD)
)

combined['mean_fraction'] = (combined['panel3_fraction'] + combined['panel4_fraction']) / 2
combined = combined.sort_values('mean_fraction', ascending=False).reset_index(drop=True)
combined.to_csv(table_out / 'stelow_patient_mhc2_classification_combined.csv', index=False)
print(combined[['PatientID', 'panel3_fraction', 'panel4_fraction', 'classification', 'discordant']].to_string(index=False))


## Brainstorming figure types

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 14))

# --- Plot 1: Connected dot plot (current) ---
ax = axes[0, 0]
for _, row in combined.iterrows():
    x = row.name
    p3 = row['panel3_fraction']
    p4 = row['panel4_fraction']
    color = 'gray' if row['discordant'] else (cmap_high_low[0] if row['classification'] == 'class II high' else cmap_high_low[1])
    ax.plot([x, x], [p3, p4], color=color, lw=1.5, alpha=0.6)
    ax.scatter(x, p3, color=color, s=80, marker='o', edgecolors=color)
    ax.scatter(x, p4, color=color, s=80, marker='o', facecolors='none', edgecolors=color, linewidths=2)
ax.axhline(MHCII_THRESHOLD, color='gray', linestyle='--', lw=1.5, alpha=0.7)
ax.set_xticks(range(len(combined)))
ax.set_xticklabels(combined['patient_label'], rotation=45, ha='right')
ax.set_ylabel('PanCK+MHCII+ fraction')
ax.set_ylim(-0.05, 1.05)
ax.set_title('1. Connected dot plot')
sns.despine(ax=ax)

# --- Plot 2: Paired stacked bars ---
ax = axes[0, 1]
bar_width = 0.35
x = np.arange(len(combined))
for i, (_, row) in enumerate(combined.iterrows()):
    color = 'gray' if row['discordant'] else (cmap_high_low[0] if row['classification'] == 'class II high' else cmap_high_low[1])
    # panel 3
    ax.bar(i - bar_width/2, row['panel3_fraction'], bar_width,
           color=color, alpha=0.9, label='Panel 3' if i == 0 else '')
    ax.bar(i - bar_width/2, 1 - row['panel3_fraction'], bar_width,
           bottom=row['panel3_fraction'], color=color, alpha=0.2)
    # panel 4
    ax.bar(i + bar_width/2, row['panel4_fraction'], bar_width,
           color=color, alpha=0.9, edgecolor='black', linewidth=1.5,
           label='Panel 4' if i == 0 else '')
    ax.bar(i + bar_width/2, 1 - row['panel4_fraction'], bar_width,
           bottom=row['panel4_fraction'], color=color, alpha=0.2,
           edgecolor='black', linewidth=1.5)
ax.axhline(MHCII_THRESHOLD, color='gray', linestyle='--', lw=1.5, alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(combined['patient_label'], rotation=45, ha='right')
ax.set_ylabel('PanCK+MHCII+ fraction')
ax.set_ylim(0, 1.05)
ax.set_title('2. Paired stacked bars')
sns.despine(ax=ax)

# --- Plot 3: Connected dot plot + shading ---
ax = axes[1, 0]
for _, row in combined.iterrows():
    x_pos = row.name
    p3 = row['panel3_fraction']
    p4 = row['panel4_fraction']
    color = 'gray' if row['discordant'] else (cmap_high_low[0] if row['classification'] == 'class II high' else cmap_high_low[1])
    # background shading showing full 0-1 range
    ax.bar(x_pos, 1, width=0.6, color=color, alpha=0.08, zorder=0)
    ax.plot([x_pos, x_pos], [p3, p4], color=color, lw=1.5, alpha=0.6)
    ax.scatter(x_pos, p3, color=color, s=80, marker='o', edgecolors=color)
    ax.scatter(x_pos, p4, color=color, s=80, marker='o', facecolors='none', edgecolors=color, linewidths=2)
ax.axhline(MHCII_THRESHOLD, color='gray', linestyle='--', lw=1.5, alpha=0.7)
ax.set_xticks(range(len(combined)))
ax.set_xticklabels(combined['patient_label'], rotation=45, ha='right')
ax.set_ylabel('PanCK+MHCII+ fraction')
ax.set_ylim(-0.05, 1.05)
ax.set_title('3. Connected dot plot + shading')
sns.despine(ax=ax)

# --- Plot 4: Slope chart ---
ax = axes[1, 1]
for _, row in combined.iterrows():
    p3 = row['panel3_fraction']
    p4 = row['panel4_fraction']
    color = 'gray' if row['discordant'] else (cmap_high_low[0] if row['classification'] == 'class II high' else cmap_high_low[1])
    ax.plot([0, 1], [p3, p4], color=color, lw=1.5, alpha=0.8, marker='o', markersize=8)
    ax.text(-0.05, p3, row['patient_label'], ha='right', va='center', fontsize=8, color=color)
    ax.text(1.05, p4, row['patient_label'], ha='left', va='center', fontsize=8, color=color)
ax.axhline(MHCII_THRESHOLD, color='gray', linestyle='--', lw=1.5, alpha=0.7)
ax.set_xlim(-0.3, 1.3)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Panel 3', 'Panel 4'], fontsize=12)
ax.set_ylabel('PanCK+MHCII+ fraction')
ax.set_ylim(-0.05, 1.05)
ax.set_title('4. Slope chart')
sns.despine(ax=ax)

plt.tight_layout()
plt.show()

## Figure 5C — per-patient MHC II+ fraction across panels

Each patient is shown as a paired dot (panel 3 and panel 4) connected by
a line. Patients are sorted by panel 4 MHC II+ fraction. The dashed line
indicates the classification threshold (0.4). S21 is highlighted in gray
to show its discordant classification across panels. Orange = class II high,
purple = class II low.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

bar_width = 0.35
mhc2_pos_color = cmap_high_low[0]  # orange
mhc2_neg_color = cmap_high_low[1]  # purple

for i, (_, row) in enumerate(combined.iterrows()):
    edge_color = 'gray' if row['discordant'] else 'none'
    edge_width = 2.5 if row['discordant'] else 0
    hatch = '//' if row['discordant'] else ''

    # panel 3 — solid filled
    ax.bar(i - bar_width/2, row['panel3_fraction'], bar_width,
           color=mhc2_pos_color,
           edgecolor=edge_color, linewidth=edge_width)
    ax.bar(i - bar_width/2, 1 - row['panel3_fraction'], bar_width,
           bottom=row['panel3_fraction'],
           color=mhc2_neg_color,
           edgecolor=edge_color, linewidth=edge_width)

    # panel 4 — hatched
    ax.bar(i + bar_width/2, row['panel4_fraction'], bar_width,
           color=mhc2_pos_color, hatch='...',
           edgecolor=edge_color if row['discordant'] else 'white',
           linewidth=edge_width if row['discordant'] else 1.5)
    ax.bar(i + bar_width/2, 1 - row['panel4_fraction'], bar_width,
           bottom=row['panel4_fraction'],
           color=mhc2_neg_color, hatch='...',
           edgecolor=edge_color if row['discordant'] else 'white',
           linewidth=edge_width if row['discordant'] else 1.5)

ax.axhline(MHCII_THRESHOLD, color='k', linestyle='--', lw=3, alpha=0.7)
ax.text(len(combined) - 0.3, MHCII_THRESHOLD + 0.02,
        f'Classification Threshold = {MHCII_THRESHOLD}', color='k', fontsize=10, ha='left')

# add background shading for classification groups
high_indices = combined[combined['classification'] == 'class II high'].index.tolist()
low_indices = combined[combined['classification'] == 'class II low'].index.tolist()
discordant_indices = combined[combined['discordant']].index.tolist()

# shade high group
ax.axvspan(min(high_indices) - 0.5, max(high_indices) + 0.5,
           alpha=0.2, color=cmap_high_low[0], zorder=0)

# shade low group  
ax.axvspan(min(low_indices) - 0.5, max(low_indices) + 0.5,
           alpha=0.2, color=cmap_high_low[1], zorder=0)

# shade discordant group with gray + hatch
ax.axvspan(min(discordant_indices) - 0.5, max(discordant_indices) + 0.5,
           alpha=0.15, color='gray', zorder=0, hatch='///',
           edgecolor='gray', linewidth=0.5)

# labels above plot
ax.text(np.mean(high_indices), 1.07, 'MHC II high',
        ha='center', va='bottom', fontsize=11,
        color=cmap_high_low[0], fontweight='bold',
        transform=ax.get_xaxis_transform())

ax.text(np.mean(low_indices), 1.07, 'MHC II low',
        ha='center', va='bottom', fontsize=11,
        color=cmap_high_low[1], fontweight='bold',
        transform=ax.get_xaxis_transform())

ax.text(np.mean(discordant_indices), 1.07, 'Discordant',
        ha='center', va='bottom', fontsize=11,
        color='gray', fontweight='bold',
        transform=ax.get_xaxis_transform())

ax.set_xticks(np.arange(len(combined)))
ax.set_xticklabels(combined['patient_label'], rotation=45, ha='center')
ax.set_ylabel('Proportion')
ax.set_ylim(0, 1.05)
ax.set_xlim(-0.5, len(combined) - 0.5)

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [
    Patch(facecolor=mhc2_pos_color, label='PanCK+MHCII+ Cells'),
    Patch(facecolor=mhc2_neg_color, label='PanCK+MHCII- Cells'),
    Patch(facecolor='white', edgecolor='black', linewidth=1.5, label='Panel 3 (filled)'),
    Patch(facecolor='white', edgecolor='black', linewidth=1.5,
          alpha=0.5, label='Panel 4 (lighter)'),
    Patch(facecolor='white', edgecolor='gray', linewidth=2.5,
          hatch='//', label='Discordant (S21)'),
]
ax.legend(handles=legend_elements, bbox_to_anchor=(1.02, 1),
          loc='upper left', frameon=False, fontsize=10)

sns.despine(ax=ax)
plt.tight_layout()
plt.savefig(fig_out / 'fig5c_stelow_mhc2_classification.pdf', bbox_inches='tight')
plt.show()